# InternImage + Rabi + Adam



In [ ]:
import os, sys, math, time, pickle, warnings, json
import copy
from pathlib import Path
from collections import OrderedDict
from functools import partial
from typing import Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_class_weight
from timm.models.layers import DropPath, trunc_normal_

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1. EXPERIMENT CONFIGURATION


SMOKE_TEST    = True
SMOKE_BATCHES = 2
SMOKE_EPOCHS  = 2

BATCH_SIZE   = 128
MAX_EPOCHS   = 100
LR           = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0
SEED         = 42
NUM_WORKERS  = 4

# Scheduler
SCHED_FACTOR   = 0.5
SCHED_PATIENCE = 8
SCHED_MIN_LR   = 1e-6

# Early stopping
ES_PATIENCE = 15

# Mixed precision
USE_AMP = True

torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = False   # False required for benchmark mode

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"PyTorch: {torch.__version__}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2. DATASET LOADING AND AUDIT

NB_DIR = Path('.').resolve()    # Models_DL/
PICKLE_PATH = NB_DIR / 'FINAL_RABI.pickle'
assert PICKLE_PATH.exists(), f"Dataset not found: {PICKLE_PATH}"

print(f"Loading {PICKLE_PATH.name} (FINAL_RABI.pickle) ...")
t0 = time.time()
with open(PICKLE_PATH, 'rb') as f:
    raw = pickle.load(f)
print(f"  Loaded in {time.time()-t0:.1f}s | type={type(raw).__name__} | keys={{'...' if len(raw)>5 else list(raw.keys())}}")

classes = sorted(raw.keys())
class_to_id = {c: i for i, c in enumerate(classes)}
id_to_class = {i: c for c, i in class_to_id.items()}
NUM_CLASSES = len(classes)
print(f"  Classes ({NUM_CLASSES}): {classes}")

arrays, labels = [], []
for c in classes:
    for s in raw[c]:
        if isinstance(s, dict) and 'data' in s:
            arrays.append(s['data'])  # ndarray (12, 7, 7) float64
            labels.append(c)
        else:
            arrays.append(s)
            labels.append(c)
del raw

total = len(arrays)
print(f"  Total samples: {total:,}")
print(f"  Sample shape:  {arrays[0].shape}, dtype={arrays[0].dtype}")


check_idx = np.random.choice(total, min(1000, total), replace=False)
nan_count = sum(np.any(np.isnan(arrays[i])) for i in check_idx)
inf_count = sum(np.any(np.isinf(arrays[i])) for i in check_idx)
print(f"  NaN check (1000 samples): {nan_count} affected | Inf: {inf_count} affected")

# Per-class counts
print("  Per-class sample counts:")
label_arr = np.array(labels)
for c in classes:
    cnt = int((label_arr == c).sum())
    print(f"    {c:30s}: {cnt:6,}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3. SPLIT LOADING, NORMALIZATION, DATASET CLASS, DATALOADERS

SPLITS_DIR = NB_DIR / 'SPLITS'

if SPLITS_DIR.exists() and (SPLITS_DIR / 'rabi_train_idx.npy').exists():
    print(f"Loading frozen splits from {SPLITS_DIR.resolve()} ...")
    train_idx = np.load(SPLITS_DIR / 'rabi_train_idx.npy')
    val_idx   = np.load(SPLITS_DIR / 'rabi_val_idx.npy')
    test_idx  = np.load(SPLITS_DIR / 'rabi_test_idx.npy')
    print(f"  Loaded: train={len(train_idx):,} / val={len(val_idx):,} / test={len(test_idx):,}")
else:
    print("WARNING: SPLITS/ folder or index files not found in Models_DL/.")
    print("         Please copy the 6 .npy split files to Models_DL/SPLITS/")
    print("         Expected files:")
    for fn in ['rabi_train_idx.npy', 'rabi_val_idx.npy', 'rabi_test_idx.npy']:
        print(f"           SPLITS/{fn}")
    print()
    print("Generating deterministic fallback splits (80/10/10, seed=42).")
    rng = np.random.RandomState(42)
    label_arr_local = np.array(labels)
    tr, va, te = [], [], []
    for c in classes:
        ci = np.where(label_arr_local == c)[0]; rng.shuffle(ci)
        n = len(ci); nt = int(round(n*0.80)); nv = int(round(n*0.10))
        tr.extend(ci[:nt]); va.extend(ci[nt:nt+nv]); te.extend(ci[nt+nv:])
    train_idx = np.array(tr, dtype=np.int64)
    val_idx   = np.array(va, dtype=np.int64)
    test_idx  = np.array(te, dtype=np.int64)
    SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    np.save(SPLITS_DIR / 'rabi_train_idx.npy', train_idx)
    np.save(SPLITS_DIR / 'rabi_val_idx.npy', val_idx)
    np.save(SPLITS_DIR / 'rabi_test_idx.npy', test_idx)
    print(f"  Saved splits to {SPLITS_DIR}")
    print(f"  Generated: train={len(train_idx):,} / val={len(val_idx):,} / test={len(test_idx):,}")


# Verify zero overlap
assert len(set(train_idx) & set(val_idx)) == 0, "TRAIN/VAL OVERLAP!"
assert len(set(train_idx) & set(test_idx)) == 0, "TRAIN/TEST OVERLAP!"
assert len(set(val_idx) & set(test_idx)) == 0, "VAL/TEST OVERLAP!"
print("  Split overlap check: PASSED")

# Normalization: per-channel z-score from TRAINING data ONLY
print("Computing train-only normalization stats ...")
train_arr = np.stack([arrays[i] for i in train_idx])  # (N_train, 12, 7, 7)
norm_mean = np.mean(train_arr, axis=(0, 2, 3))         # (12,)
norm_std  = np.std(train_arr, axis=(0, 2, 3))          # (12,)
norm_std  = np.where(norm_std < 1e-6, 1.0, norm_std)
del train_arr
print(f"  mean: {norm_mean.round(4)}")
print(f"  std:  {norm_std.round(4)}")


class CropDataset(Dataset):
    '''
    Satellite tile dataset with optional spatial augmentation.
    Valid augmentations for 7x7 satellite patches (scientifically justified):
      - hflip, vflip: fields are spatially symmetric (no preferred orientation)
      - rot90: satellite view from above — rotation invariant
    NOT applied: color jitter, blur, erase — would corrupt spectral bands.
    augment=True only for training set.
    '''
    def __init__(self, indices, augment=False):
        self.indices = indices
        self.augment = augment
        self.mean_t = torch.tensor(norm_mean, dtype=torch.float32).view(12, 1, 1)
        self.std_t  = torch.tensor(norm_std,  dtype=torch.float32).view(12, 1, 1)
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        tensor = torch.tensor(arrays[real_idx], dtype=torch.float32)  # (12, 7, 7)
        tensor = (tensor - self.mean_t) / (self.std_t + 1e-7)
        if self.augment:
            if torch.rand(1).item() > 0.5:
                tensor = torch.flip(tensor, dims=[2])           # hflip
            if torch.rand(1).item() > 0.5:
                tensor = torch.flip(tensor, dims=[1])           # vflip
            k = torch.randint(0, 4, (1,)).item()
            if k > 0:
                tensor = torch.rot90(tensor, k=k, dims=[1, 2])  # rot90
        label  = class_to_id[labels[real_idx]]
        return tensor, torch.tensor(label, dtype=torch.long)

BS = BATCH_SIZE if not SMOKE_TEST else 4
train_loader = DataLoader(CropDataset(train_idx, augment=(not SMOKE_TEST)), batch_size=BS, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'), drop_last=True)
val_loader   = DataLoader(CropDataset(val_idx),   batch_size=BS, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
test_loader  = DataLoader(CropDataset(test_idx),  batch_size=BS, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
print(f"DataLoaders ready: batch_size={BS}, num_workers={NUM_WORKERS}")
xb, yb = next(iter(train_loader))
print(f"  Batch shape: x={xb.shape}, y={yb.shape}, y_range=[{yb.min().item()}, {yb.max().item()}]")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4. MODEL ARCHITECTURE — InternImage


class _to_channels_first(nn.Module):
    def forward(self, x): return x.permute(0, 3, 1, 2)

class _to_channels_last(nn.Module):
    def forward(self, x): return x.permute(0, 2, 3, 1)

def _build_norm_layer(dim, norm_layer, in_format='channels_last', out_format='channels_last', eps=1e-6):
    layers = []
    if norm_layer == 'BN':
        if in_format == 'channels_last': layers.append(_to_channels_first())
        layers.append(nn.BatchNorm2d(dim))
        if out_format == 'channels_last': layers.append(_to_channels_last())
    elif norm_layer == 'LN':
        if in_format == 'channels_first': layers.append(_to_channels_last())
        layers.append(nn.LayerNorm(dim, eps=eps))
        if out_format == 'channels_first': layers.append(_to_channels_first())
    return nn.Sequential(*layers)

def _build_act_layer(act_layer):
    if act_layer == 'ReLU': return nn.ReLU(inplace=True)
    if act_layer == 'SiLU': return nn.SiLU(inplace=True)
    if act_layer == 'GELU': return nn.GELU()
    raise NotImplementedError(f'build_act_layer does not support {act_layer}')


def _get_reference_points(spatial_shapes, device, kernel_h, kernel_w, dilation_h, dilation_w, pad_h=0, pad_w=0, stride_h=1, stride_w=1):
    _, H_, W_, _ = spatial_shapes
    H_out = (H_ - (dilation_h * (kernel_h - 1) + 1)) // stride_h + 1
    W_out = (W_ - (dilation_w * (kernel_w - 1) + 1)) // stride_w + 1
    ref_y, ref_x = torch.meshgrid(
        torch.linspace((dilation_h*(kernel_h-1))//2+0.5, (dilation_h*(kernel_h-1))//2+0.5+(H_out-1)*stride_h, H_out, dtype=torch.float32, device=device),
        torch.linspace((dilation_w*(kernel_w-1))//2+0.5, (dilation_w*(kernel_w-1))//2+0.5+(W_out-1)*stride_w, W_out, dtype=torch.float32, device=device),
    )
    ref_y = ref_y.reshape(-1)[None] / H_
    ref_x = ref_x.reshape(-1)[None] / W_
    return torch.stack((ref_x, ref_y), -1).reshape(1, H_out, W_out, 1, 2)


def _generate_dilation_grids(spatial_shapes, kernel_h, kernel_w, dilation_h, dilation_w, group, device):
    _, H_, W_, _ = spatial_shapes
    x, y = torch.meshgrid(
        torch.linspace(-((dilation_w*(kernel_w-1))//2), -((dilation_w*(kernel_w-1))//2)+(kernel_w-1)*dilation_w, kernel_w, dtype=torch.float32, device=device),
        torch.linspace(-((dilation_h*(kernel_h-1))//2), -((dilation_h*(kernel_h-1))//2)+(kernel_h-1)*dilation_h, kernel_h, dtype=torch.float32, device=device),
    )
    grid = torch.stack([x/W_, y/H_], -1).reshape(-1, 1, 2).repeat(1, group, 1).permute(1, 0, 2)
    return grid.reshape(1, 1, 1, group * kernel_h * kernel_w, 2)


def dcnv3_core_pytorch(input, offset, mask, kernel_h, kernel_w, stride_h, stride_w, pad_h, pad_w, dilation_h, dilation_w, group, group_channels, offset_scale, remove_center=False):
    input = F.pad(input, [0, 0, pad_h, pad_h, pad_w, pad_w])
    N_, H_in, W_in, _ = input.shape
    _, H_out, W_out, _ = offset.shape
    ref  = _get_reference_points(input.shape, input.device, kernel_h, kernel_w, dilation_h, dilation_w, pad_h, pad_w, stride_h, stride_w)
    grid = _generate_dilation_grids(input.shape, kernel_h, kernel_w, dilation_h, dilation_w, group, input.device)
    spatial_norm = torch.tensor([W_in, H_in]).reshape(1,1,1,2).repeat(1,1,1,group*kernel_h*kernel_w).to(input.device)
    P_ = kernel_h * kernel_w
    sampling_locations = (ref + grid * offset_scale).repeat(N_, 1, 1, 1, 1).flatten(3, 4)
    sampling_locations = sampling_locations + offset * offset_scale / spatial_norm
    sampling_grids = 2 * sampling_locations - 1
    input_ = input.view(N_, H_in*W_in, group*group_channels).transpose(1,2).reshape(N_*group, group_channels, H_in, W_in)
    sampling_grid_ = sampling_grids.view(N_, H_out*W_out, group, P_, 2).transpose(1,2).flatten(0,1)
    sampling_input_ = F.grid_sample(input_, sampling_grid_, mode='bilinear', padding_mode='zeros', align_corners=False)
    mask = mask.view(N_, H_out*W_out, group, P_).transpose(1,2).reshape(N_*group, 1, H_out*W_out, P_)
    output = (sampling_input_ * mask).sum(-1).view(N_, group*group_channels, H_out*W_out)
    return output.transpose(1,2).reshape(N_, H_out, W_out, -1).contiguous()


class DCNv3_pytorch(nn.Module):
    """
    DCNv3 pure-PyTorch implementation.
    Source: official InternImage / ops_dcnv3 / modules / dcnv3.py
    Functionally identical to the CUDA-compiled version; no make.sh needed.
    """
    def __init__(self, channels=64, kernel_size=3, dw_kernel_size=None, stride=1, pad=1,
                 dilation=1, group=4, offset_scale=1.0, act_layer='GELU', norm_layer='LN',
                 center_feature_scale=False, remove_center=False, **kwargs):
        super().__init__()
        if channels % group != 0:
            raise ValueError(f'channels ({channels}) must be divisible by group ({group})')
        self.channels = channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.dilation = dilation
        self.pad = pad
        self.group = group
        self.group_channels = channels // group
        self.offset_scale = offset_scale
        self.remove_center = int(remove_center)
        self.center_feature_scale = center_feature_scale
        if dw_kernel_size is not None:
            self.dw_conv = nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size=dw_kernel_size, stride=1,
                          padding=(dw_kernel_size-1)//2, groups=channels),
                _build_norm_layer(channels, 'LN', in_format='channels_first', out_format='channels_last'),
                _build_act_layer(act_layer)
            )
        self.offset = nn.Linear(channels, group * kernel_size * kernel_size * 2)
        self.mask   = nn.Linear(channels, group * kernel_size * kernel_size)
        self.input_proj = nn.Linear(channels, channels)
        self.output_proj = nn.Linear(channels, channels)
        self._reset_parameters()
        if center_feature_scale:
            self.center_feature_scale_proj_weight = nn.Parameter(torch.zeros((group, channels), dtype=torch.float))
            self.center_feature_scale_proj_bias   = nn.Parameter(torch.tensor(0.0, dtype=torch.float).view((1,)).repeat(group))
            self.center_feature_scale_module = nn.Sequential(nn.Linear(channels, group))

    def _reset_parameters(self):
        nn.init.constant_(self.offset.weight.data, 0.)
        nn.init.constant_(self.offset.bias.data, 0.)
        nn.init.constant_(self.mask.weight.data, 0.)
        nn.init.constant_(self.mask.bias.data, 0.)
        nn.init.xavier_uniform_(self.input_proj.weight.data)
        nn.init.constant_(self.input_proj.bias.data, 0.)
        nn.init.xavier_uniform_(self.output_proj.weight.data)
        nn.init.constant_(self.output_proj.bias.data, 0.)

    def forward(self, input):
        N, H, W, _ = input.shape
        x = self.input_proj(input)
        dtype = x.dtype
        if hasattr(self, 'dw_conv'):
            x1 = x.permute(0,3,1,2)
            x1 = self.dw_conv(x1)
            offset = self.offset(x1)
            mask   = self.mask(x1).reshape(N,H,W,self.group*self.kernel_size*self.kernel_size)
        else:
            offset = self.offset(x)
            mask   = self.mask(x).reshape(N,H,W,self.group*self.kernel_size*self.kernel_size)
        mask = F.softmax(mask, -1).reshape(N,H,W,self.group,self.kernel_size*self.kernel_size)
        x = dcnv3_core_pytorch(
            x, offset, mask,
            self.kernel_size, self.kernel_size,
            self.stride, self.stride,
            self.pad, self.pad,
            self.dilation, self.dilation,
            self.group, self.group_channels,
            self.offset_scale,
        )
        if self.center_feature_scale:
            center_feature_scale = self.center_feature_scale_module(F.linear(input, self.center_feature_scale_proj_weight, self.center_feature_scale_proj_bias)).sigmoid().view(N,H,W,self.group,1).repeat(1,1,1,1,self.group_channels).flatten(-2)
            x = x * (1 - center_feature_scale) + input.view(N,H,W,self.channels) * center_feature_scale
        x = self.output_proj(x)
        return x


# ════════════════════════════════════════════════════════════════
# InternImage Model Classes
class _MLPLayer(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer='GELU', drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1  = nn.Linear(in_features, hidden_features)
        self.act  = _build_act_layer(act_layer)
        self.fc2  = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class _InternImageLayer(nn.Module):
    def __init__(self, core_op, channels, groups, mlp_ratio=4., drop=0., drop_path=0.,
                 act_layer='GELU', norm_layer='LN', post_norm=False, layer_scale=None,
                 offset_scale=1.0, with_cp=False, dw_kernel_size=None,
                 res_post_norm=False, center_feature_scale=False, remove_center=False):
        super().__init__()
        self.with_cp = with_cp
        self.post_norm = post_norm
        self.norm1 = _build_norm_layer(channels, 'LN')
        self.dcn   = core_op(channels=channels, kernel_size=3, stride=1, pad=1, dilation=1,
                             group=groups, offset_scale=offset_scale, act_layer=act_layer,
                             norm_layer=norm_layer, dw_kernel_size=dw_kernel_size,
                             center_feature_scale=center_feature_scale, remove_center=remove_center)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = _build_norm_layer(channels, 'LN')
        self.mlp   = _MLPLayer(in_features=channels, hidden_features=int(channels*mlp_ratio), act_layer=act_layer, drop=drop)
        self.layer_scale = layer_scale is not None
        if self.layer_scale:
            self.gamma1 = nn.Parameter(layer_scale * torch.ones(channels), requires_grad=True)
            self.gamma2 = nn.Parameter(layer_scale * torch.ones(channels), requires_grad=True)

    def forward(self, x):
        def _inner(x):
            if not self.layer_scale:
                if self.post_norm:
                    x = x + self.drop_path(self.norm1(self.dcn(x)))
                    x = x + self.drop_path(self.norm2(self.mlp(x)))
                else:
                    x = x + self.drop_path(self.dcn(self.norm1(x)))
                    x = x + self.drop_path(self.mlp(self.norm2(x)))
            else:
                if self.post_norm:
                    x = x + self.drop_path(self.gamma1 * self.norm1(self.dcn(x)))
                    x = x + self.drop_path(self.gamma2 * self.norm2(self.mlp(x)))
                else:
                    x = x + self.drop_path(self.gamma1 * self.dcn(self.norm1(x)))
                    x = x + self.drop_path(self.gamma2 * self.mlp(self.norm2(x)))
            return x
        return checkpoint.checkpoint(_inner, x) if (self.with_cp and x.requires_grad) else _inner(x)


class _DownsampleLayer(nn.Module):
    def __init__(self, channels, norm_layer='LN'):
        super().__init__()
        self.conv = nn.Conv2d(channels, 2*channels, kernel_size=3, stride=2, padding=1, bias=False)
        self.norm = _build_norm_layer(2*channels, norm_layer, 'channels_first', 'channels_last')
    def forward(self, x):
        return self.norm(self.conv(x.permute(0,3,1,2)))


class _InternImageBlock(nn.Module):
    """
    One stage of InternImage = N InternImageLayer blocks + optional downsample.
    Source: official InternImage repo / classification / models / intern_image.py
    """
    def __init__(self, core_op, channels, depth, groups, downsample=True, mlp_ratio=4.,
                 drop=0., drop_path=0., act_layer='GELU', norm_layer='LN', post_norm=False,
                 offset_scale=1.0, layer_scale=None, with_cp=False, dw_kernel_size=None,
                 post_norm_block_ids=None, res_post_norm=False, center_feature_scale=False,
                 remove_center=False):
        super().__init__()
        self.post_norm = post_norm
        self.center_feature_scale = center_feature_scale
        self.blocks = nn.ModuleList([
            _InternImageLayer(
                core_op=core_op, channels=channels, groups=groups, mlp_ratio=mlp_ratio, drop=drop,
                drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                act_layer=act_layer, norm_layer=norm_layer, post_norm=post_norm,
                layer_scale=layer_scale, offset_scale=offset_scale, with_cp=with_cp,
                dw_kernel_size=dw_kernel_size, res_post_norm=res_post_norm,
                center_feature_scale=center_feature_scale, remove_center=remove_center,
            ) for i in range(depth)
        ])
        if not self.post_norm or center_feature_scale:
            self.norm = _build_norm_layer(channels, 'LN')
        self.downsample = _DownsampleLayer(channels=channels, norm_layer=norm_layer) if downsample else None

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        if not self.post_norm or self.center_feature_scale:
            x = self.norm(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x



class InputAdapter12(nn.Module):
    \"\"\"
    Improved 12-ch adapter. Two-branch: spatial + scalar.
    out_ch=12 avoids the 12->3 bottleneck.
    \"\"\"
    def __init__(self, in_ch=12, out_ch=12, scalar_chs=(8,9,10,11)):
        super().__init__()
        self.scalar_chs = list(scalar_chs)
        self.spatial_conv = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.spatial_bn   = nn.BatchNorm2d(out_ch)
        self.scalar_proj  = nn.Linear(len(scalar_chs), out_ch, bias=True)
        self.act = nn.GELU()
        self.blend = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        B, C, H, W = x.shape
        s  = self.act(self.spatial_bn(self.spatial_conv(x)))
        sc = x[:, self.scalar_chs, :, :].mean(dim=(-1,-2))
        sc = self.act(self.scalar_proj(sc)).unsqueeze(-1).unsqueeze(-1).expand_as(s)
        w = torch.sigmoid(self.blend)
        return w * s + (1 - w) * sc


class MicroStemLayer(nn.Module):
    """
    Stride-1 stem for 7x7 inputs (deviation from official 2x stride-2 stem).
    Official 2x stride-2 would collapse 7x7 -> 1x1 before any stage, losing all spatial info.
    Conv2d(3->64, 3x3, stride=1, pad=1) + BN + GELU -> channels_last (B,H,W,C).
    """
    def __init__(self, in_ch=3, out_ch=64):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, 1, 1, bias=False)
        self.norm = nn.BatchNorm2d(out_ch)
        self.act  = nn.GELU()
    def forward(self, x):
        return self.act(self.norm(self.conv(x))).permute(0, 2, 3, 1)  # (B,H,W,C)


class InternImageCropClassifier(nn.Module):
    """
    InternImage-based crop classifier for (B, 12, 7, 7) satellite inputs.
    Architecture (spatial flow):
      InputAdapter: (B,12,7,7) -> (B,3,7,7)
      MicroStem:    (B,3,7,7)  -> (B,7,7,64)   [channels_last]
      Stage 0:      ch=64,  grp=1,  depth=1  -> 7x7  -> downsample -> 4x4
      Stage 1:      ch=128, grp=2,  depth=1  -> 4x4  -> downsample -> 2x2
      Stage 2:      ch=256, grp=4,  depth=1  -> 2x2  -> downsample -> 1x1
      Stage 3:      ch=512, grp=8,  depth=1  -> 1x1  (no downsample)
      permute -> (B,512,1,1) -> conv_head(512->768) + BN + GELU
      AvgPool -> Flatten -> Linear(768, num_classes)
    """
    def __init__(self, num_classes, in_chans=12, channels=64,
                 depths=None, groups=None, drop_path_rate=0.1, cls_scale=1.5):
        super().__init__()
        if depths is None: depths = [1, 1, 1, 1]
        if groups  is None: groups  = [1, 2, 4, 8]
        num_levels = len(depths)
        num_features = int(channels * 2 ** (num_levels - 1))  # 512
        head_dim     = int(num_features * cls_scale)           # 768
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]

        self.adapter     = InputAdapter12(in_chans, in_chans)  # 12->12
        self.patch_embed = MicroStemLayer(in_chans, channels)
        self.pos_drop    = nn.Dropout(p=0.0)
        self.levels = nn.ModuleList()
        for i in range(num_levels):
            self.levels.append(_InternImageBlock(
                core_op=DCNv3_pytorch, channels=int(channels * 2**i),
                depth=depths[i], groups=groups[i], mlp_ratio=4.0, drop=0.0,
                drop_path=dpr[sum(depths[:i]):sum(depths[:i+1])],
                act_layer='GELU', norm_layer='LN', post_norm=False,
                downsample=(i < num_levels - 1), layer_scale=None,
                offset_scale=1.0, with_cp=False,
            ))
        self.conv_head = nn.Sequential(
            nn.Conv2d(num_features, head_dim, 1, bias=False),
            nn.BatchNorm2d(head_dim), nn.GELU(),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.head    = nn.Linear(head_dim, num_classes)

    def forward(self, x):
        x = self.adapter(x)      # (B,3,7,7)
        x = self.patch_embed(x)  # (B,7,7,64)
        x = self.pos_drop(x)
        for lvl in self.levels: x = lvl(x)
        x = x.permute(0, 3, 1, 2)  # (B,512,1,1)
        x = self.conv_head(x)
        x = self.avgpool(x).flatten(1)
        return self.head(x)


# Instantiate
model = InternImageCropClassifier(num_classes=NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"InternImageCropClassifier | Classes={NUM_CLASSES} | Params={train_params:,} total={total_params:,}")

try:
    dummy = torch.zeros(2, 12, 7, 7).to(DEVICE)
    with torch.no_grad(): out = model(dummy)
    print(f"Shape test PASSED: input={dummy.shape} -> output={out.shape}")
except Exception as e:
    print(f"Shape test FAILED: {e}")

# Optimizer: Adam, no weight decay for offset/mask/bias (DCNv3 params)
no_wd = [p for n, p in model.named_parameters() if p.requires_grad and ('bias' in n or 'offset' in n or 'mask' in n)]
wd    = [p for n, p in model.named_parameters() if p.requires_grad and not ('bias' in n or 'offset' in n or 'mask' in n)]
optimizer = torch.optim.Adam(
    [{'params': wd,    'weight_decay': WEIGHT_DECAY},
     {'params': no_wd, 'weight_decay': 0.0}],
    lr=LR, betas=(0.9, 0.999), eps=1e-8
)
print(f"Optimizer: Adam | lr={LR} | wd={WEIGHT_DECAY} | no-wd params={len(no_wd)} | wd params={len(wd)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5. TRAINING LOOP WITH ADAM

RESULTS_DIR = NB_DIR / 'Results' / 'Rabi/InternImage'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = RESULTS_DIR / 'best_model.pth'

# Class-weighted CrossEntropyLoss
train_label_ids = np.array([class_to_id[labels[i]] for i in train_idx])
cw_np = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_label_ids)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw_np, dtype=torch.float32).to(DEVICE))
print(f"Loss: CrossEntropyLoss with balanced class weights")

# cuDNN benchmark: fixed input size => auto-tunes convolution kernels
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True

# LR Scheduler: ReduceLROnPlateau on val_loss
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=SCHED_FACTOR, patience=SCHED_PATIENCE,
    min_lr=SCHED_MIN_LR, verbose=False
)
print(f"Scheduler: ReduceLROnPlateau | factor={SCHED_FACTOR} | patience={SCHED_PATIENCE} | min_lr={SCHED_MIN_LR}")

# AMP scaler (no-op on CPU or when USE_AMP=False)
_use_amp = USE_AMP and (DEVICE.type == 'cuda')
scaler = torch.cuda.amp.GradScaler(enabled=_use_amp)
print(f"AMP mixed-precision: {_use_amp}")

# Early stopping state
_es_counter = 0
_es_best_loss = float('inf')
_stopped_early = False

# Training state
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
best_val_acc = 0.0
training_start = time.time()

EPOCHS = SMOKE_EPOCHS if SMOKE_TEST else MAX_EPOCHS
print(f"Training for {EPOCHS} epoch(s) | SMOKE_TEST={SMOKE_TEST} | Device={DEVICE}")

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────
    model.train()
    ep_loss, ep_correct, ep_total = 0.0, 0, 0
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        if SMOKE_TEST and batch_idx >= SMOKE_BATCHES:
            break
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=_use_amp):
            outputs = model(inputs)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        if GRAD_CLIP > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        ep_loss    += loss.item() * inputs.size(0)
        ep_correct += (outputs.argmax(1) == targets).sum().item()
        ep_total   += inputs.size(0)
    tr_loss = ep_loss / ep_total
    tr_acc  = ep_correct / ep_total

    # ── Validate ─────────────────────────────────────────────
    model.eval()
    va_loss, va_correct, va_total = 0.0, 0, 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(val_loader):
            if SMOKE_TEST and batch_idx >= SMOKE_BATCHES:
                break
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=_use_amp):
                outputs = model(inputs)
                loss = criterion(outputs, targets)
            va_loss    += loss.item() * inputs.size(0)
            va_correct += (outputs.argmax(1) == targets).sum().item()
            va_total   += inputs.size(0)
    val_loss = va_loss / va_total
    val_acc  = va_correct / va_total

    # ── LR Warmup + Scheduler step ───────────────────────────
    if not SMOKE_TEST:
        if epoch <= LR_WARMUP_EPOCHS:
            warmup_lr = LR * (epoch / LR_WARMUP_EPOCHS)
            for pg in optimizer.param_groups:
                pg['lr'] = warmup_lr
        else:
            scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    # ── Save best checkpoint by val_acc ──────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
            'scheduler_state_dict': scheduler.state_dict(),
            'classes': classes,
            'class_to_id': class_to_id,
            'norm_mean': norm_mean.tolist(),
            'norm_std': norm_std.tolist(),
        }, CKPT_PATH)

    # ── Early Stopping (monitored: val_loss) ──────────────────
    if not SMOKE_TEST:
        if val_loss < _es_best_loss - 1e-6:
            _es_best_loss = val_loss
            _es_counter = 0
        else:
            _es_counter += 1
        if _es_counter >= ES_PATIENCE:
            elapsed = time.time() - training_start
            print(f"Epoch {epoch:3d}/{EPOCHS} | "
                  f"Train Loss={tr_loss:.4f} Acc={tr_acc:.4f} | "
                  f"Val Loss={val_loss:.4f} Acc={val_acc:.4f} | "
                  f"Best Val Acc={best_val_acc:.4f} | LR={current_lr:.2e} | {elapsed:.0f}s")
            print(f"
Early stopping at epoch {epoch} (no val_loss improvement for {ES_PATIENCE} epochs)")
            _stopped_early = True
            break

    elapsed = time.time() - training_start
    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"Train Loss={tr_loss:.4f} Acc={tr_acc:.4f} | "
          f"Val Loss={val_loss:.4f} Acc={val_acc:.4f} | "
          f"Best Val Acc={best_val_acc:.4f} | LR={current_lr:.2e} | {elapsed:.0f}s")

print(f"\nTraining complete. Best val_acc={best_val_acc:.4f} | Stopped early: {_stopped_early}")
print(f"Checkpoint saved to: {CKPT_PATH}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 6. EVALUATION ON TEST SET

if not CKPT_PATH.exists():
    print("WARNING: No checkpoint found (smoke test may not have saved). Evaluating current weights.")
else:
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded best checkpoint: epoch={ckpt['epoch']}, val_acc={ckpt['val_acc']:.4f}")

model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for batch_idx, (inputs, targets) in enumerate(test_loader):
        if SMOKE_TEST and batch_idx >= SMOKE_BATCHES:
            break
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(targets.numpy())

all_preds   = np.array(all_preds)
all_targets = np.array(all_targets)

test_acc      = accuracy_score(all_targets, all_preds)
test_prec_mac = precision_score(all_targets, all_preds, average='macro', zero_division=0)
test_rec_mac  = recall_score(all_targets, all_preds, average='macro', zero_division=0)
test_f1_mac   = f1_score(all_targets, all_preds, average='macro', zero_division=0)
test_f1_wtd   = f1_score(all_targets, all_preds, average='weighted', zero_division=0)

print(f"\nTest Results {'(SMOKE_TEST — limited batches)' if SMOKE_TEST else ''}:")
print(f"  Accuracy:        {test_acc:.4f}")
print(f"  Precision (mac): {test_prec_mac:.4f}")
print(f"  Recall (mac):    {test_rec_mac:.4f}")
print(f"  F1 (macro):      {test_f1_mac:.4f}")
print(f"  F1 (weighted):   {test_f1_wtd:.4f}")
print()
print(classification_report(all_targets, all_preds, target_names=classes, zero_division=0))

In [ ]:
# ─────────────────────────────────────────────────────────────
# 7. RESULT PLOTS
# ─────────────────────────────────────────────────────────────
SMOKE_TAG = ' [SMOKE TEST]' if SMOKE_TEST else ''
epochs_x = range(1, len(history['train_loss'])+1)

# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_x, history['train_loss'], 'b-', label='Train Loss')
axes[0].plot(epochs_x, history['val_loss'], 'r-', label='Val Loss')
axes[0].set(title=f'Loss Curves{SMOKE_TAG}', xlabel='Epoch', ylabel='Loss')
axes[0].legend(); axes[0].grid(True)
axes[1].plot(epochs_x, history['train_acc'], 'b-', label='Train Acc')
axes[1].plot(epochs_x, history['val_acc'], 'r-', label='Val Acc')
axes[1].set(title=f'Accuracy Curves{SMOKE_TAG}', xlabel='Epoch', ylabel='Accuracy')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=120, bbox_inches='tight')
plt.show(); print(f"Saved: training_curves.png")

# Confusion matrix
cm = confusion_matrix(all_targets, all_preds)
fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES), max(6, NUM_CLASSES)))
sns.heatmap(cm, annot=(NUM_CLASSES <= 20), fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=ax, linewidths=0.3)
ax.set(title=f'Confusion Matrix{SMOKE_TAG}', xlabel='Predicted', ylabel='True')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show(); print(f"Saved: confusion_matrix.png")

# Per-class accuracy
per_class_acc = []
for i in range(NUM_CLASSES):
    mask = all_targets == i
    if mask.sum() > 0:
        per_class_acc.append(float((all_preds[mask] == all_targets[mask]).mean()))
    else:
        per_class_acc.append(0.0)
fig, ax = plt.subplots(figsize=(max(10, NUM_CLASSES*0.7), 5))
bars = ax.bar(classes, per_class_acc, color='steelblue', edgecolor='black', linewidth=0.5)
ax.set(title=f'Per-Class Accuracy{SMOKE_TAG}', ylabel='Accuracy', ylim=(0, 1.05))
ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=8)
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, f'{acc:.2f}', ha='center', va='bottom', fontsize=6)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'per_class_accuracy.png', dpi=120, bbox_inches='tight')
plt.show(); print(f"Saved: per_class_accuracy.png")

# Learning rate curve
if 'lr' in history and len(history['lr']) > 0:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.semilogy(range(1, len(history['lr'])+1), history['lr'], 'g-', linewidth=1.5)
    ax.set(title=f'Learning Rate{SMOKE_TAG}', xlabel='Epoch', ylabel='LR (log scale)')
    ax.grid(True)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'lr_curve.png', dpi=120, bbox_inches='tight')
    plt.show(); print(f"Saved: lr_curve.png")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 8. FINAL EXPERIMENT REPORT
# ─────────────────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
duration = time.time() - training_start
gpu_mem = f"{torch.cuda.max_memory_allocated()/1e9:.2f} GB" if torch.cuda.is_available() else "N/A (CPU)"

print("=" * 65)
print(f"  EXPERIMENT: InternImage + Rabi + Adam")
print("=" * 65)
print(f"  Dataset:          {PICKLE_PATH.name}")
print(f"  Classes:          {NUM_CLASSES}")
print(f"  Train / Val / Test: {len(train_idx):,} / {len(val_idx):,} / {len(test_idx):,}")
print(f"  Batch size:       {BS}")
print(f"  Epochs trained:   {len(history['train_loss'])}")
print(f"  Best val acc:     {best_val_acc:.4f}")
print(f"  Test accuracy:    {test_acc:.4f}")
print(f"  Test F1 (macro):  {test_f1_mac:.4f}")
print(f"  Test F1 (wtd):    {test_f1_wtd:.4f}")
print(f"  Total params:     {total_params:,}")
print(f"  Trainable params: {train_params:,}")
print(f"  Device:           {DEVICE}")
print(f"  Peak GPU memory:  {gpu_mem}")
print(f"  Training duration: {duration/60:.1f} min")
print(f"  Checkpoint:       {CKPT_PATH}")
print(f"  Results dir:      {RESULTS_DIR}")
print(f"  SMOKE_TEST:       {SMOKE_TEST}")
print("=" * 65)
if SMOKE_TEST:
    print()
    print("  *** SMOKE TEST COMPLETE — set SMOKE_TEST=False for full training ***")
    print("  *** On GPU: set FORWARD_TYPE='v2' (VMamba) and compile DCNv3      ***")